# Session 4: Chemical data, formats, and structures

A gas-phase infrared spectrum of acetone from the NIST Chemistry WebBook is in `data/acetone_ir.jdx` as a JCAMP-DX file: a text file whose header says what was measured and whose 88 data lines hold 880 absorbances in a compressed form. Section 1 reads it as a document, then as a dataset, and checks the dataset against the document. Section 2 does the same for a molecule assigned to you, downloaded from the same WebBook in class. Section 3 reads the data block a second way, the way a first attempt often does, and shows what the checks catch. Section 4 is the judgment cell, scored for the first time today.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 4 assignment on Canvas by 1:30 pm on Thursday, October 15. Submissions are individual from today. Keep your downloaded file in `data/` under the name the WebBook gave it, `<CAS number>-IR.jdx`; the graders run your notebook against the same files under the same names. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports the standard library module `sys`, NumPy and Matplotlib, which is all this notebook uses. Reading the file itself takes nothing beyond Python's own `open()`. The cell prints nothing when it works.

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt

## Parameters

The committed file and your own. `ACETONE_FILE` is a relative path, as in session 3, from this folder. `MY_CAS` is the CAS registry number of the molecule assigned to you in section 2, with its hyphens, as the board gives it; leave it as `CHANGE-ME` until then. `MAXY_TOLERANCE` is how far the largest parsed value may sit from the header's `MAXY` before the check fails; the header gives `MAXY` to five decimals, so 0.001 is loose enough for rounding and tight enough to catch a wrong reading.

In [ ]:
ACETONE_FILE = "data/acetone_ir.jdx"
MY_CAS = "CHANGE-ME"   # your assigned molecule, section 2
MAXY_TOLERANCE = 0.001   # in the file's y units; the largest parsed value against the header's MAXY

## 1. Explore: a JCAMP-DX file read as a document

To know what a plot of this file will show, read the file before plotting it. JCAMP-DX is the exchange format that spectrometer software has written since 1988, and a JCAMP-DX file is plain text, so `open()` reads it. The next cell reads every line into a list and prints the first 12 with `repr`, which shows the line ending on each and any stray spaces that a plain `print` would hide.

In [ ]:
with open(ACETONE_FILE) as f:
    lines = f.readlines()

print(len(lines), "lines")
for line in lines[:12]:
    print(repr(line))

Every line that starts with `##` is a header field, `##NAME=value`, and the two lines under `##OWNER=` that do not start with `##` continue the owner's copyright statement. The header runs to `##XYDATA=(X++(Y..Y))`, which names the form of the data block that follows, and the file ends with `##END=`. The next cell turns the header into a dictionary by slicing off the `##` and splitting each line at its first `=`, the two operations from the opening, and once it reaches `XYDATA` it keeps the lines that follow as the data block. It prints the field count (which includes `END`), the data line count, the five fields that fix what the numbers mean, and `RESOLUTION`, which this file lacks, all fetched with `.get()`, which returns `None` for an absent field where `header["RESOLUTION"]` would raise `KeyError`.

In [ ]:
header = {}
data_lines = []
in_data = False
for line in lines:
    line = line.strip()
    if line.startswith("##"):
        key, value = line[2:].split("=", 1)
        header[key] = value
        in_data = key == "XYDATA"
    elif in_data and line:
        data_lines.append(line)

print(len(header), "header fields and", len(data_lines), "data lines")
for key in ["XUNITS", "YUNITS", "DELTAX", "YFACTOR", "NPOINTS", "RESOLUTION"]:
    print(f"{key:10s} {header.get(key)}")

`NPOINTS` says 880 and there are 88 data lines, so a line is not a point. The form `(X++(Y..Y))` means that each line starts with one wavenumber and continues with the absorbances at that wavenumber and at the next ones, spaced by `DELTAX`, 4 per cm, ten to a line. The absorbances are stored as integers that `YFACTOR` scales, so the 40 on the first line is 40 times 0.000078659 = 0.00315 AU, which is the header's `FIRSTY`. The next cell expands the block into two arrays. Every value in the file is text until `float()` converts it, the header's included.

In [ ]:
delta_x = float(header["DELTAX"])
y_factor = float(header["YFACTOR"])
x_factor = float(header.get("XFACTOR", "1"))

x = []
y = []
for line in data_lines:
    fields = line.split()
    x_start = float(fields[0]) * x_factor
    for k in range(1, len(fields)):
        x.append(x_start + (k - 1) * delta_x)
        y.append(float(fields[k]) * y_factor)
x = np.array(x)
y = np.array(y)

print(len(x), "points from", x[0], "to", x[-1], header["XUNITS"])
print("first absorbance", round(y[0], 6), "AU, header FIRSTY", header["FIRSTY"])

To see the spectrum, plot absorbance against wavenumber with the wavenumber axis reversed, the convention for infrared spectra, so that the high wavenumbers are on the left. One series, so no legend.

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, y)
ax.invert_xaxis()
ax.set_xlabel(f"wavenumber ({header['XUNITS']})")
ax.set_ylabel("absorbance (AU)")
ax.set_title(f"{header['TITLE']}, gas phase, NIST Chemistry WebBook")
fig.tight_layout()
plt.show()

The spectrum runs from 450 to 3966 per cm and its strongest band, 0.787 AU at 1738 per cm, is the carbonyl stretch; the next bands, at 1218 and 1366 per cm, are below 0.5 AU, the C-H stretch at 2970 per cm, the bending band at 1434 per cm and the skeletal band at 526 per cm are near 0.12 AU, and the baseline sits at zero between them. The next cell finds the strongest band from the arrays rather than by eye: `argmax` returns the index of the largest absorbance, and that index reads the wavenumber.

In [ ]:
i_max = y.argmax()
print(f"strongest band: {x[i_max]:.0f} {header['XUNITS']} at {y[i_max]:.4f} AU")

The header carries three numbers that the arrays must reproduce: `NPOINTS`, the count; `LASTX`, the last wavenumber; and `MAXY`, the largest absorbance. To check a reading of the block, compare each against the arrays. The next cell prints `ok` or `MISMATCH` for each. A MISMATCH on any of the three means the block was read wrongly, whatever the plot looks like.

In [ ]:
n_points = int(header["NPOINTS"])
last_x = float(header["LASTX"])
max_y = float(header["MAXY"])


def verdict(condition):
    """Return 'ok' when a check passes and 'MISMATCH' when it does not."""
    return "ok" if condition else "MISMATCH"


print(f"count    {len(y):>10d}   NPOINTS {n_points:>10d}   {verdict(len(y) == n_points)}")
print(f"last x   {x[-1]:>10.2f}   LASTX   {last_x:>10.2f}   {verdict(abs(x[-1] - last_x) < delta_x)}")
print(f"largest  {y.max():>10.5f}   MAXY    {max_y:>10.5f}   {verdict(abs(y.max() - max_y) < MAXY_TOLERANCE)}")

**Explain.** The header says 880 points and the file has 88 data lines. In two sentences, say what one data line holds, and which two header fields turned it into ten points on the plot, naming what each field does.

*Your answer:*

Sections 2 and 3 read another file the same way, so the next cell wraps the steps above in two functions, with docstrings as in session 2: `read_jcamp(path)` returns the header dictionary and the two arrays, and `check_jcamp(header, x, y)` prints the three checks. A function that is defined is not yet run.

In [ ]:
def read_jcamp(path):
    """Read a JCAMP-DX file whose data block is in (X++(Y..Y)) form.

    Parameters
    ----------
    path : str
        Path to the file, relative to this notebook.

    Returns
    -------
    header : dict
        Every ##NAME=value field, values as text.
    x : numpy.ndarray
        Positions in the file's XUNITS (per cm, or micrometers).
    y : numpy.ndarray
        Values in the file's YUNITS (absorbance or transmittance), YFACTOR applied.
    """
    with open(path) as f:
        lines = f.readlines()
    header = {}
    data_lines = []
    in_data = False
    for line in lines:
        line = line.strip()
        if line.startswith("##"):
            key, value = line[2:].split("=", 1)
            header[key] = value
            in_data = key == "XYDATA"
        elif in_data and line:
            data_lines.append(line)
    delta_x = float(header["DELTAX"])
    y_factor = float(header["YFACTOR"])
    x_factor = float(header.get("XFACTOR", "1"))
    x = []
    y = []
    for line in data_lines:
        fields = line.split()
        x_start = float(fields[0]) * x_factor
        for k in range(1, len(fields)):
            x.append(x_start + (k - 1) * delta_x)
            y.append(float(fields[k]) * y_factor)
    return header, np.array(x), np.array(y)


def check_jcamp(header, x, y):
    """Print the three header checks, NPOINTS, LASTX and MAXY, against the arrays.

    Parameters
    ----------
    header : dict
        The header from read_jcamp.
    x, y : numpy.ndarray
        The arrays from read_jcamp, in the file's own units.

    Returns
    -------
    None
        Prints one line per check, ending in ok or MISMATCH.
    """
    n_points = int(header["NPOINTS"])
    last_x = float(header["LASTX"])
    max_y = float(header["MAXY"])
    delta_x = float(header["DELTAX"])

    def verdict(condition):
        return "ok" if condition else "MISMATCH"

    print(f"count    {len(y):>10d}   NPOINTS {n_points:>10d}   {verdict(len(y) == n_points)}")
    print(f"last x   {x[-1]:>10.2f}   LASTX   {last_x:>10.2f}   {verdict(abs(x[-1] - last_x) < delta_x)}")
    print(f"largest  {y.max():>10.5f}   MAXY    {max_y:>10.5f}   {verdict(abs(y.max() - max_y) < MAXY_TOLERANCE)}")

The next cell runs both functions on acetone and should print the same three `ok` lines as before.

In [ ]:
header, x, y = read_jcamp(ACETONE_FILE)
check_jcamp(header, x, y)

## 2. Apply: your assigned molecule

Each of you has one molecule from the list on the board, with its CAS registry number. To get its gas-phase infrared spectrum, set `MY_CAS` in Parameters and run the next cell, which prints the WebBook address of the JCAMP-DX file for that molecule and the path the notebook will read. Open the address in your browser once; it downloads a file named `<CAS number>-IR.jdx`, the same link the compound's WebBook page offers under its IR spectrum. Download it once: the WebBook limits repeated requests from one place, and if it refuses, the same file is in the Session 4 folder of Canvas Files under the same name. Check that the saved file is named exactly as the cell prints, since a second download in some browsers becomes `...(1).jdx`, then drag it from your Downloads folder onto the `data` folder of this session in the VS Code Explorer, so that it sits at `signals-to-insights/sessions/S04-chemical-data-formats/data/` in your copy of the repository. The address is built from the number with its hyphens removed, which is what the WebBook's own links do.

In [ ]:
MY_FILE = "data/" + MY_CAS + "-IR.jdx"
print("download:", "https://webbook.nist.gov/cgi/cbook.cgi?JCAMP=C" + MY_CAS.replace("-", "") + "&Index=0&Type=IR")
print("save as: ", MY_FILE)

**[TASK 1]** Download the file as described above and put it in `data/`. The next cell reads it with `read_jcamp` and prints its title and the same fields as section 1. Until the file is there, the cell stops with `FileNotFoundError`, which is what Run All should do until the task is done.

In [ ]:
my_header, my_x, my_y = read_jcamp(MY_FILE)
print(my_header["TITLE"])
for key in ["XUNITS", "YUNITS", "DELTAX", "YFACTOR", "NPOINTS", "RESOLUTION"]:
    print(f"{key:10s} {my_header.get(key)}")

**[TASK 2]** Run the three checks on your file. The stub names the function; you supply its three arguments. Until you do, the cell stops with a `TypeError` about missing arguments.

In [ ]:
check_jcamp(...)   # [TASK 2] the header and the two arrays from your file

**Ask the assistant** before the next task, in Claude Code, with a prompt of your own that asks how it would find the strongest band in your file, for example `How do I find the strongest band in my spectrum from the my_x and my_y arrays?`. Read its answer against the `YUNITS` line that TASK 1 printed, and against your chemistry: in an absorbance file the strongest band is the largest value, whereas in a transmittance file it is the smallest, because absorbance is minus the logarithm base ten of transmittance, and the band absorbs most where the least light passes. What it suggested and what you did with it is one candidate for part (b) of the judgment cell.

**[TASK 3]** Find the strongest band. The stub leaves the index to you: `argmax` and `argmin` return the index of the largest and of the smallest value, and which one is the band depends on `YUNITS`. The cell reports the band in the file's own units, then as absorbance for a transmittance file, and as a wavenumber for a file whose `XUNITS` are micrometers, since the wavenumber in per cm is 10,000 divided by the wavelength in micrometers; it also prints the array's largest and smallest values, so you can see that your band is one of the two. Until the index is filled in, the cell stops with a `TypeError` about an ellipsis.

In [ ]:
i_band = int(...)   # [TASK 3] the index of the strongest band in my_y; read YUNITS first
band_x = my_x[i_band]
band_y = my_y[i_band]

print(f"strongest band: {band_x:.1f} {my_header['XUNITS']} at {band_y:.4f} {my_header['YUNITS'].lower()}")
print(f"the array's largest value is {my_y.max():.4f} and its smallest {my_y.min():.4f}")
if my_header["YUNITS"] == "TRANSMITTANCE":
    if band_y > 0:
        print(f"as absorbance:  {-np.log10(band_y):.3f} AU")
    else:
        print("transmittance 0 at the band: saturated, absorbance undefined")
if my_header["XUNITS"] == "MICROMETERS":
    print(f"as wavenumber:  {10000 / band_x:.1f} 1/CM")

The next cell plots your spectrum as section 1 did. The markdown cell after it is yours to write: one sentence that gives the range, the strongest band with its value, and the other bands a reader would take from the plot, as the acetone sentence did. That sentence is what makes the figure readable by someone who cannot see it.

In [ ]:
if my_header["XUNITS"] == "MICROMETERS":
    x_label = "wavelength (MICROMETERS)"
else:
    x_label = f"wavenumber ({my_header['XUNITS']})"
if my_header["YUNITS"] == "ABSORBANCE":
    y_label = "absorbance (AU)"
else:
    y_label = "transmittance (fraction)"

fig, ax = plt.subplots()
ax.plot(my_x, my_y)
if my_header["XUNITS"] != "MICROMETERS":
    ax.invert_xaxis()   # high wavenumbers on the left; a wavelength axis already runs that way
ax.set_xlabel(x_label)
ax.set_ylabel(y_label)
ax.set_title(f"{my_header['TITLE']}, gas phase, NIST Chemistry WebBook")
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**Explain.** Compare with a neighbor: give both strongest bands with the functional group each belongs to, and say whether your two files agree on `YUNITS` and `DELTAX`. If they do not, say what you had to do differently.

*Your answer:*

## 3. Analyze: two readings of one data block

Before running the next cell, ask the assistant this, or a prompt in your own words that asks the same: `Load the data block of my JCAMP-DX file with np.loadtxt, skipping the header, and tell me how many points it gives and what its largest value is.` Note what it proposes and what numbers it reports, because the next cell does the reading that `np.loadtxt` on the block amounts to, each data line as one point with the first number as x and the second as y, in a loop, then plots it over the section 2 reading and prints the strongest band each reading gives next to the three header values. The two series differ by marker: a line for the expanded reading, black circles for one point per line. The assistant's answer against this printout is the other candidate for part (b).

In [ ]:
x_line = []
y_line = []
with open(MY_FILE) as f:
    my_lines = f.readlines()
in_block = False
for line in my_lines:
    line = line.strip()
    if line.startswith("##"):
        in_block = line[2:].split("=", 1)[0] == "XYDATA"
    elif in_block and line:
        fields = line.split()
        x_line.append(float(fields[0]))
        y_line.append(float(fields[1]) * float(my_header["YFACTOR"]))
x_line = np.array(x_line)
y_line = np.array(y_line)

fig, ax = plt.subplots()
ax.plot(my_x, my_y, label=f"expanded, {len(my_y)} points")
ax.plot(x_line, y_line, "o", color="black", markersize=3, label=f"one point per line, {len(y_line)} points")
if my_header["XUNITS"] != "MICROMETERS":
    ax.invert_xaxis()
ax.set_xlabel(x_label)
ax.set_ylabel(y_label)
ax.set_title(f"{my_header['TITLE']}: two readings of the same data block")
ax.legend()
fig.tight_layout()
plt.show()

if my_header["YUNITS"] == "ABSORBANCE":
    j_band = y_line.argmax()
else:
    j_band = y_line.argmin()
print(f"expanded reading:     {band_x:.1f} at {band_y:.4f}, {len(my_y)} points, last x {my_x[-1]:.2f}, largest {my_y.max():.5f}")
print(f"one point per line:   {x_line[j_band]:.1f} at {y_line[j_band]:.4f}, {len(y_line)} points, last x {x_line[-1]:.2f}, largest {y_line.max():.5f}")
print(f"header:               NPOINTS {my_header['NPOINTS']}, LASTX {my_header['LASTX']}, MAXY {my_header['MAXY']}")

*Write one sentence here on what the two series show for your molecule.*

**Explain.** Using `NPOINTS`, `LASTX` and `MAXY`, say which reading is the spectrum that was measured, and report the wavenumber and value of the strongest band under each reading. This is the evidence that part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): report the wavenumber and the absorbance or transmittance of the strongest band in your assigned spectrum, name the header field you needed to get each of the two numbers right, and say what you would have reported for each if you had ignored that field. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your file. The suggestion can concern anything in sections 2 and 3, and the reason has to cite a number or a field from your file, not a general rule.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"Matplotlib {matplotlib.__version__}")